In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore")
    
import os, sys, glob
import json
import re
import numpy as np
import pandas as pd
from natsort import natsorted
from manual_spellchecker import spell_checker
import inflect
from scipy import stats

sys.path.append('/dartfs/rc/lab/F/FinnLab/tommy/isc_asynchrony_behavior/code/utils/')

from config import *
import dataset_utils as utils
from tommy_utils import nlp, statistics
from preproc_utils import load_model_results, divide_nwp_dataframe
import analysis_utils as analysis

## Functions -- cleaning, aggregation, and analysis

### Check and clean meta file

In [ ]:
# importing shutil module  
import shutil
from pathlib import Path

def check_used_files(experiment_name, experiment_version, task, clean_errors=False, max_missing_responses=5, video=False):
    '''
    Grabs used files based on the meta file and results directory
    Returns list of subjects that were used and ones that had errors
    '''
    
    checker = {
        'complete': [],
        'incomplete': [],
        'error': [],
        'missing': [],
    }

    meta_dir = os.path.join('/dartfs/rc/lab/F/FinnLab/tommy/jspsych_experiments/utils/experiment_meta/', experiment_name)
    results_dir = os.path.join(BASE_DIR, 'experiments',  experiment_name, 'results', experiment_version)

    if video:
        meta_file = pd.read_csv(os.path.join(meta_dir, f'{experiment_version}-{task}_video.csv'))
        source_dir = os.path.join(BASE_DIR, 'stimuli',  'presentation_orders', experiment_version, task, 'jspsych-video')
    else:
        meta_file = pd.read_csv(os.path.join(meta_dir, f'{experiment_version}-{task}.csv'))
        source_dir = os.path.join(BASE_DIR, 'stimuli',  'presentation_orders', experiment_version, task, 'jspsych')
    
    # grab the used files
    used_fns = meta_file[meta_file['used'].fillna(1).astype(bool)]
    
    approve_ids = []
    
    # go through each used file
    for i, fn in used_fns.iterrows():
        # grab info regarding subject name and modality
        curr_path = Path(fn['subject_fns'])
        sub = curr_path.stem.split('_')[0]
        modality = curr_path.parents[0].stem
        
        # find the corresponding parameter file for the current subject
        parameter_fn = glob.glob(os.path.join(source_dir, f'{sub}*.json'))
        assert (len(parameter_fn) == 1)
        
        # load the parameter file to compare to the subject's results
        df_parameters = pd.read_json(parameter_fn[0], orient='records')
        df_parameters = df_parameters.dropna()
        df_parameters['word_index'] = df_parameters['word_index'].dropna().astype(int)
        
        # then grab the subject results
        sub_results_dir = os.path.join(results_dir, task, modality, sub)

        # load results from the completed experiment
        try:
            # Responses being trimmed by one for video is now accounted for in load_participant_results
            current_id, demographics, experience, responses = analysis.load_participant_results(sub_results_dir, sub)
            
            # append if approving
            approve_ids.append(current_id)
        except:
#             if os.path.exists(sub_results_dir):
            checker['error'].append((i, modality, sub, None))
            continue

        # #The last trial is logged within video but not the other two conditions (FOR NOW)
        # if video:
        #     # Responses being trimmed is now accounted for in load_participant_results
        #     df_parameters = df_parameters.iloc[:-1]

        # return responses, df_parameters
        
        # check that all indices of trials match and all responses are there
        all_trials_complete = np.all(responses['word_index'] == df_parameters['word_index'])
        missing_response_threshold = sum(pd.isnull(responses['response'])) <= max_missing_responses
        
#         all_responses_complete = np.all(~pd.isnull(responses['response']))
        
        # also ensure that we have the right amount of demographics/experience questions
        all_checks_complete = np.all([
            all_trials_complete, 
            missing_response_threshold, 
            len(demographics)==4,
            len(experience)==2,
        ])
        
        if all_trials_complete and missing_response_threshold:
            # add to list of people completed
            checker['complete'].append((i, modality, sub, current_id))
        else:
            checker['incomplete'].append((i, modality, sub, current_id))
            
        del current_id
        
    if clean_errors:
        clean_meta_errors(checker, experiment_name, experiment_version, task, video=video)
        
        # run again and return from here now that its updated
        return check_used_files(experiment_name, experiment_version, task, clean_errors=False, video=video)
    else:
        return checker, approve_ids

def clean_meta_errors(checker, experiment_name, experiment_version, task, video=False):

    meta_dir = os.path.join('/dartfs/rc/lab/F/FinnLab/tommy/jspsych_experiments/utils/experiment_meta/', experiment_name)

    if video:
        meta_fn = os.path.join(meta_dir, f'{experiment_version}-{task}_video.csv')
    else:
        meta_fn = os.path.join(meta_dir, f'{experiment_version}-{task}.csv')   

    meta_file = pd.read_csv(meta_fn)
    
    results_dir = os.path.join(BASE_DIR, 'experiments',  experiment_name, 'results', experiment_version)

    # Also include the incomplete ones
    errors = checker['error'] + checker['incomplete']
    modalities = ['video', 'text', 'audio']
    
    if any(errors):
        
        remove_idxs, _, _, _ = zip(*errors) 
        remove_idxs = list(remove_idxs)
        
        for modality in modalities:

            # get errors for the current modality
            modality_errors = [error for error in errors if error[1] == modality]
            
            errors_dir = os.path.join(results_dir, task, modality, 'error')

            # get new errors dir if previous has files in it
            batch_errors = sorted(glob.glob(os.path.join(errors_dir, '*')))

            if any(batch_errors):
                last_error_dir = Path(batch_errors[-1]).stem
            else:
                last_error_dir = 'batch_1'

            if any(glob.glob(os.path.join(errors_dir, last_error_dir, '*'))):
                curr_batch_num = int(last_error_dir.split('_')[-1]) + 1
                curr_error_dir = os.path.join(errors_dir, f'batch_{curr_batch_num}')
                os.makedirs(curr_error_dir)
            else:
                curr_error_dir = os.path.join(errors_dir, last_error_dir)

            for item in modality_errors:
                file_idx, modality, sub, prolific_id = item

                # then grab the subject results
                sub_results_dir = os.path.join(results_dir, task, modality, sub)

                if os.path.exists(sub_results_dir):
                    shutil.move(sub_results_dir, curr_error_dir)


        print (f'Cleaned meta file!')
        meta_file.loc[list(remove_idxs), 'used'] = None
        meta_file['used'] = meta_file['used'].astype('Int64')
        meta_file.to_csv(meta_fn, index=False)


### Interactive data cleaning

In [ ]:
import enchant, string, time
from IPython.display import clear_output

AUTO_REPLACE = {
    'mum': 'mom',
    'mums': 'mom',
    'okay': 'ok',
    'sh': 'she',
    'barefoote': 'barefoot',
    'cellphone': 'phone',
    'fag': 'cigarette',
    'lite': 'light',
    'yea': 'yes',
    'yeah': 'yes',
    # 'want': 'wanna',
}

def clean_participant_responses(df_results, df_transcript, video=False):
    
    # grab indices of responses --> used to index back in
    response_indices = df_results['experiment_phase'] == 'test'
    response_indices = np.where(response_indices)[0]

    checked_indices = []

    # Need to remove the final row
    if video:
        df_results.loc[response_indices[-1], ['response', 'experiment_phase']] = False
        response_indices = response_indices[:-1]
        
    # filter down to get the responses
    df_responses = df_results.iloc[response_indices, :].reset_index(drop=True)

    df_responses.loc[df_responses['response'] == False, 'response'] = ""
    df_responses['response'] = df_responses['response'].apply(lambda x: x.strip().lower())

    inflect_engine = inflect.engine()
    enc_dict = enchant.Dict("en_US")

    ##############################
    ###### Run autoreplace #######
    ##############################

    for k, v in AUTO_REPLACE.items():
        df_responses.loc[df_responses['response'] == k, 'response'] = v

    ##############################
    #### Run numbers replace #####
    ##############################

    for index, df in df_responses.iterrows():
        response = df['response']
         
        if response.isdigit():
            response = inflect_engine.number_to_words(response)
            df['response'] = response
            df_responses.iloc[df.name] = df
            checked_indices.append(index)

    ##############################
    ###### Run spell-check #######
    ##############################

    print (f'##########################\n' +
           f'### Running spellcheck ###\n' +
           f'##########################\n\n')

    time.sleep(2)

    for index, df in df_responses.iterrows():

        response = df['response']
        
        if response == '':
            continue
        
        # tokens = df['response'].split()
        if not enc_dict.check(response) or response in string.punctuation and index not in checked_indices:
            df = prompt_correct_response(df, df_transcript, enc_dict, prompt_correction=False)
            df_responses.iloc[df.name] = df
            checked_indices.append(index)
            

    ##############################
    ######## Find phrases ########
    ##############################

    clear_output(wait=False)
    print (f'##########################\n' +
           f'####### Find phrases #####\n' +
           f'##########################\n\n')

    time.sleep(2)

    # go through each row
    for index, df in df_responses.iterrows():
        response = df['response'].split()

        if len(response) > 1 and index not in checked_indices:
            df = prompt_correct_response(df, df_transcript, enc_dict, prompt_correction=False)
            df_responses.iloc[df.name] = df
            checked_indices.append(index)
        else:
            continue

    ##############################
    ######## Final check #########
    ##############################

    clear_output(wait=False)
    print (f'##########################\n' +
           f'####### Final check ######\n' +
           f'##########################\n\n')

    time.sleep(2)

    # go through each row
    for index, df in df_responses.iterrows():

        if index not in checked_indices:
            df = prompt_correct_response(df, df_transcript, enc_dict, prompt_correction=True)
            df_responses.iloc[df.name] = df
            checked_indices.append(index)
        
    df_results.iloc[response_indices] = df_responses

    return df_results

def prompt_correct_response(df_response, df_transcript, enc_dict, range_display=7, prompt_correction=False):
    
    word_index = df_response['word_index']
    response = df_response['response']
    ground_truth = df_response['critical_word']
    
    start_index = (word_index - range_display) if (word_index - range_display) >= 0 else 0 
    end_index = (word_index + range_display) if (word_index + range_display) - len(df_transcript) <= 0 else None
    
    # grab the context    
    start_context = df_transcript['Word_Written'].iloc[start_index:word_index]
    end_context = df_transcript['Word_Written'].iloc[word_index + 1:end_index]
    
    # display the word
    string_to_print = ""
    
    if start_index > 0:
        string_to_print+= ".... "
    
    string_to_print+= " ".join(start_context) + " " + "\033[43;30m" + response + "\033[m" + " " + " ".join(end_context)
    
    if end_index is not None and end_index < len(df_transcript):
        string_to_print+= " ...."

    clear_output(wait=False)

    print("\n\nCurrent Word: " + string_to_print)
    print ("Ground Truth: ", ground_truth)
    
    # suggestions = enc_dict.suggest(misspelled_word)
    if prompt_correction:
        prompt_correction = input('\nNeeds correction? [y/n]: ')
    
    if prompt_correction == 'y' or prompt_correction == False:
        suggestions = enc_dict.suggest(response)
        print("\Suggestions: ", suggestions)
        correct_word = input("\nCorrect Version: ")
        
        if correct_word == "-999":
            break_flag = True
            sys.exit(0)
        elif correct_word.isdigit() and int(correct_word) - 1 < len(suggestions): # User wants to use suggestion
            print (f'Using word: {suggestions[int(correct_word)-1]}')
            df_response['response'] = df_response['response'].replace(response, suggestions[int(correct_word)-1])
            time.sleep(2)
        elif len(correct_word) == 0: # User wants to Skip
            return df_response
        elif correct_word == "''" or correct_word == '""': # User wants to remove the word
            df_response['response'] = df_response['response'].replace(response, "")
        else:
            time.sleep(2)
            df_response['response'] = df_response['response'].replace(response, correct_word)

    return df_response

## Data cleaning

In [ ]:
EXPERIMENT_NAME = 'next-word-prediction'
EXPERIMENT_VERSION = 'pilot-multimodal-01'
TASK = 'howtodraw'
video = True

# set the directories we need
gentle_dir = os.path.join(BASE_DIR, 'stimuli/gentle')
results_dir = os.path.join(BASE_DIR, 'experiments',  EXPERIMENT_NAME, 'results', EXPERIMENT_VERSION)
preproc_dir = os.path.join(BASE_DIR, 'stimuli/preprocessed')
models_dir = os.path.join(BASE_DIR, 'derivatives/model-predictions')

# make directories
cleaned_results_dir = os.path.join(BASE_DIR, 'experiments',  EXPERIMENT_NAME, 'cleaned-results', EXPERIMENT_VERSION)
behavioral_dir = os.path.join(BASE_DIR, 'derivatives/results/behavioral/')
stim_dir = os.path.join(BASE_DIR, f'stimuli/cut_audio/{EXPERIMENT_VERSION}')

### Clean meta file and remove errors

In [ ]:
# check all the files
checker, approve_ids = check_used_files(EXPERIMENT_NAME, EXPERIMENT_VERSION, TASK, clean_errors=True, video=video)
approve_ids = list(map(str, approve_ids))

# find the subject list based on complete files
file_idx, modality, sub_list, prolific_ids = zip(*checker['complete'])
sub_mod_list = list(zip(sub_list, modality))

print (len(set(approve_ids)))
print ('\n'.join(approve_ids))

#### Specific modification for stimulus black

In [ ]:
import os
import datetime
def modification_date(filename):
    t = os.path.getmtime(filename)
    return datetime.datetime.fromtimestamp(t)


all_indices = []

for i, (sub, mod) in enumerate(sub_mod_list):
    sub_results_dir = os.path.join(results_dir, TASK, mod, sub)
    info = modification_date(os.path.join(sub_results_dir, f'{sub}_next-word-prediction.csv'))

    if info.year == 2024:
        all_indices.append(i)

current_ids = [approve_ids[idx] for idx in all_indices]
print ('\n'.join(current_ids))

### Manually spell-check participants data

In [ ]:
ADJUSTED_WORDS = {
    'wanna': lambda x: np.logical_and(x['response'] == 'want', x['critical_word'] == 'wanna'),
    'gonna': lambda x: np.logical_and(x['response'] == 'going', x['critical_word'] == 'gonna'),
}


df_transcript = pd.read_csv(os.path.join(preproc_dir, TASK, f'{TASK}_transcript-preprocessed.csv'))

for i, (sub, modality) in enumerate(sub_mod_list):


    sub_cleaned_dir = os.path.join(cleaned_results_dir, TASK, modality, sub)
    out_fn = os.path.join(sub_cleaned_dir, f'{sub}_next-word-prediction.csv')
    
    utils.attempt_makedirs(sub_cleaned_dir)

    ############################
    #### Load subject data #####
    ############################

    if os.path.exists(out_fn):
        print (f'File exists: {modality} {sub}')
        df_results = pd.read_csv(out_fn)
        
        for k, v in AUTO_REPLACE.items():
            df_results.loc[df_results['response'] == k, 'response'] = v

        for word, word_filter in ADJUSTED_WORDS.items():
            current_filter = word_filter(df_results)
            word_filters = np.logical_and(df_results['response'] == 'want', df_results['critical_word'] == 'wanna')
        
            if np.any(current_filter):
                df_results.loc[current_filter, 'response'] = word
        
        df_results.to_csv(out_fn, index=False)

        print (f'Saved file for {modality} {sub}')
        continue
    else:
        print (f'Correcting: {modality} {sub}')
        # load and filter down to response trials 
        sub_dir = os.path.join(results_dir, TASK, modality, sub)
        df_results = pd.read_csv(os.path.join(sub_dir, f'{sub}_next-word-prediction.csv')).fillna(False)

    ############################
    ###### Check responses #####
    ############################

    
    df_results['word_index'] = df_results['word_index'].astype(int)
    df_results = clean_participant_responses(df_results, df_transcript, video=video)
    
    ############################
    #### Save cleaned data #####
    ############################
    
    df_results.to_csv(out_fn, index=False)

    print (f'Saved file for {modality} {sub}')
    # if os.path.exists(sub_dir):
    #     current_id, demographics, expserience, responses = load_participant_results(sub_dir, sub)
    #     responses['response'] = responses['response'].fillna('')

## Compile data across participants

### Load word models for semantic comparisons

In [ ]:
word_model_name = 'fasttext'
word_model = nlp.load_word_model(model_name=word_model_name, cache_dir=CACHE_DIR)
word_model_info = (word_model_name, word_model)

### Load all participants data and save file

In [ ]:
stim_dir = os.path.join(BASE_DIR, f'stimuli/cut_audio/{EXPERIMENT_VERSION}')

# load transcript
df_transcript = pd.read_csv(os.path.join(preproc_dir, TASK, f'{TASK}_transcript-preprocessed.csv'))
df_transcript = df_transcript.rename(columns={'Word_Written': 'word', 'Punctuation': 'punctuation'})

# load all results, calculate accuracy, then save
df_all_results = aggregate_participant_responses(cleaned_results_dir, stim_dir, TASK, sub_mod_list, n_orders=4 if TASK == 'black' else 3)
# df_all_results, df_all_accuracy = calculate_results_accuracy(df_all_results)

# save compiled cleaned results
out_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-cleaned-behavior.csv')
df_all_results.to_csv(out_fn, index=False)

### Lemmatize data (responses + ground truth) and save 

In [ ]:
# use transcript to lemmatize responses, then calculate accuracy
df_lemmatized_results = lemmatize_responses(df_all_results.copy(), df_transcript)
df_lemmatized_results, df_lemmatized_accuracy = calculate_results_accuracy(df_lemmatized_results)

# save compiled lemmatized results
out_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-cleaned-behavior_lemmatized.csv')
df_lemmatized_results.to_csv(out_fn, index=False)

### Analyze human data and save

In [ ]:
# load transcript
df_transcript = pd.read_csv(os.path.join(preproc_dir, TASK, f'{TASK}_transcript-preprocessed.csv'))
df_transcript = df_transcript.rename(columns={'Word_Written': 'word', 'Punctuation': 'punctuation'})

# load lemmatized file and analyze human results --> use all words
lemmatized_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-cleaned-behavior_lemmatized.csv')
df_lemmatized_results = pd.read_csv(lemmatized_fn)

# combine the data and lemmatize model results
df_analyzed_lemmatized = analyze_human_results(df_transcript, df_lemmatized_results, word_model_info, top_n=None, drop_rt=None)

# save lemmatized human-model results
out_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-analyzed-behavior_human-lemmatized.csv')
df_analyzed_lemmatized.to_csv(out_fn, index=False)

## Analyze and compile human and LLM data

### Set LLM model names

In [ ]:
PROSODY_MODELS = [
    'helsinki-prosody_scratch-gpt2_joint-loss_prosody-embed',
    'helsinki-prosody_scratch-gpt2_clm-loss_prosody-embed',
    'helsinki-prosody_scratch-gpt2_clm-loss_no-prosody-embed',

    ################### GIGASPEECH ############
    
    'gigaspeech-prosody_scratch-gpt2_joint-loss_prosody-embed',
    'gigaspeech-prosody_scratch-gpt2_clm-loss_prosody-embed',
    'gigaspeech-prosody_scratch-gpt2_clm-loss_no-prosody-embed',
]

model_names = PROSODY_MODELS


# # get all MLM models except BERT
# MLM_MODELS = list(nlp.MLM_MODELS_DICT.keys())[1:]
# CLM_MODELS = list(nlp.CLM_MODELS_DICT.keys()) 
# model_names = CLM_MODELS + MLM_MODELS

# print (f'Loading the following models')
# print (f'MLM models: {MLM_MODELS}')
# print (f'CLM models: {CLM_MODELS}')

### Cleaned responses - Load and merge human and LLM data 

In [ ]:
# load lemmatized file and analyze human results --> use all words
cleaned_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-cleaned-behavior.csv')
df_cleaned_results = pd.read_csv(cleaned_fn)

# combine the data and lemmatize model results
df_human_model = compare_human_model_accuracy(df_cleaned_results, model_names, word_model_info, task=TASK, top_n=1, window_size=25, lemmatize=False)

# save combined human-model results
out_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-analyzed-behavior_human-model.csv')
df_human_model.to_csv(out_fn, index=False)

### Lemmatization - Load and merge human and LLM data

In [ ]:
WINDOW_SIZE = 100

# load lemmatized file and analyze human results --> use all words
lemmatized_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-cleaned-behavior_lemmatized.csv')
df_lemmatized_results = pd.read_csv(lemmatized_fn)

# combine the data and lemmatize model results
df_human_model_lemmatized = compare_human_model_accuracy(df_lemmatized_results, model_names, word_model_info, task=TASK, top_n=1, window_size=WINDOW_SIZE, lemmatize=True)

# save lemmatized human-model results
if 'prosody' in model_names[0]:
    out_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-analyzed-behavior_window-size-{WINDOW_SIZE}_human-prosody-model-lemmatized.csv')
else:
    out_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-analyzed-behavior_window-size-{WINDOW_SIZE}_human-model-lemmatized.csv')

df_human_model_lemmatized.to_csv(out_fn, index=False)

## Analyze human vs. GPT2-XL distributions

### Load file of cleaned data

In [ ]:
# save lemmatized human-model results
human_cleaned_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-cleaned-behavior_lemmatized.csv')
df_human_cleaned = pd.read_csv(human_cleaned_fn)

### Compare human and model prediction distributions and save

In [ ]:
WINDOW_SIZE = 25

df_comparison = []
df_human_cleaned['response'] = df_human_cleaned['response'].apply(lambda x: strip_punctuation(x) if isinstance(x, str) else '')
    
for model_name in model_names:

    print (f'{model_name}')

    # now load a model to compare to 
    tokenizer, model = nlp.load_clm_model(
        model_name='gpt2' if 'prosody' in model_name else model_name, 
        cache_dir=CACHE_DIR)
    
    # go through the each modality word index
    for (modality, response_index), df in df_human_cleaned.groupby(['modality', 'word_index']):
        
        # get all responses for current index across both modalities
        all_responses = df_human_cleaned[df_human_cleaned['word_index'] == response_index]['response'].tolist()
        all_responses = list(filter(None, all_responses))
    
        # grab responses for the current modality
        modality_responses = df['response'].apply(strip_punctuation)
        ground_truth = df['ground_truth'].unique().tolist()[0]
        
        # prosody models can't be run with less than 4 tokens
        if 'prosody' in model_name and response_index < 4:
            continue

        # load the logits for the current response
        model_logits = load_logits(models_dir, model_name, TASK, WINDOW_SIZE, response_index) #response_index - 1)
        
        # now compare the two and add it to the dataframe
        df_compare = compare_human_model_distributions(tokenizer, word_model, modality_responses, all_responses, model_logits, ground_truth)
        df_compare['model_name'] = model_name
        df_compare['modality'] = modality
        df_compare['word_index'] = response_index
        df_compare[['entropy_group', 'accuracy_group', 'ground_truth']] = df[['entropy_group', 'accuracy_group', 'ground_truth']].iloc[0]
        
        df_comparison.append(df_compare)

# lastly 
df_comparison = pd.concat(df_comparison).reset_index(drop=True)

if 'prosody' in model_names[0]:
    out_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-analyzed-behavior_window-size-{WINDOW_SIZE}_human-prosody-model-distributions-lemmatized.csv')
else:
    out_fn = os.path.join(behavioral_dir, f'task-{TASK}_group-analyzed-behavior_window-size-{WINDOW_SIZE}_human-model-distributions-lemmatized.csv')

df_comparison.to_csv(out_fn, index=False)